In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from PIL import Image

dataset_path = "/Users/yusufabdulkadir/Downloads/CO2Wounds-V2 Extended Chronic Wounds Dataset From Leprosy Patients-2/imgs"  # <<<<<< CHANGE THIS
image_size = 64
batch_size = 64
nz = 100
num_epochs = 50
lr = 0.0002
beta1 = 0.5
ngpu = 1


class ImageFolderNoClass(Dataset):
    def __init__(self, root_dir, transform=None):
        self.files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = ImageFolderNoClass(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


class Generator(nn.Module):
    def __init__(self, ngpu):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)


device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
netG = Generator(ngpu).to(device)
netD = Discriminator(ngpu).to(device)
criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

print("Starting Training Loop...")
for epoch in range(num_epochs):
    for i, real_images in enumerate(dataloader, 0):
        real_images = real_images.to(device)
        b_size = real_images.size(0)

        # Train D
        netD.zero_grad()
        label = torch.full((b_size,), 1., device=device)
        output = netD(real_images).view(-1)
        lossD_real = criterion(output, label)
        lossD_real.backward()

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_images = netG(noise)
        label.fill_(0.)
        output = netD(fake_images.detach()).view(-1)
        lossD_fake = criterion(output, label)
        lossD_fake.backward()
        optimizerD.step()

        # Train G
        netG.zero_grad()
        label.fill_(1.)
        output = netD(fake_images).view(-1)
        lossG = criterion(output, label)
        lossG.backward()
        optimizerG.step()

        if i % 50 == 0:
            print(f"[{epoch}/{num_epochs}][{i}/{len(dataloader)}] "
                  f"Loss_D: {(lossD_real.item()+lossD_fake.item()):.4f} "
                  f"Loss_G: {lossG.item():.4f}")

    # Save generated samples
    with torch.no_grad():
        fake = netG(fixed_noise).detach().cpu()
    utils.save_image(fake, f"epoch_{epoch}.png", normalize=True)

torch.save(netG.state_dict(), "generator.pth")
torch.save(netD.state_dict(), "discriminator.pth")
print("Training complete! Models saved.")


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import utils
from PIL import Image


nz = 100          # latent vector size
image_size = 64   # image size used in training
ngpu = 1
num_images = 1000
batch_size = 64   # generate in batches to avoid memory issues
output_dir = "generated_images"


class Generator(nn.Module):
    def __init__(self, ngpu):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)


device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
netG = Generator(ngpu).to(device)
netG.load_state_dict(torch.load("generator.pth", map_location=device))
netG.eval()

# Create output directory
os.makedirs(output_dir, exist_ok=True)

print(f"Generating {num_images} images...")

with torch.no_grad():
    generated_count = 0
    
    while generated_count < num_images:
        # Calculate batch size for this iteration
        current_batch_size = min(batch_size, num_images - generated_count)
        
        # Generate random noise
        noise = torch.randn(current_batch_size, nz, 1, 1, device=device)
        
        # Generate fake images
        fake_images = netG(noise)
        
        # Save each image individually
        for i in range(current_batch_size):
            # Convert tensor to PIL image
            img_tensor = fake_images[i].cpu()
            # Normalize from [-1, 1] to [0, 1]
            img_tensor = (img_tensor + 1) / 2
            img_tensor = torch.clamp(img_tensor, 0, 1)
            
            # Save image
            filename = f"{output_dir}/generated_{generated_count:04d}.png"
            utils.save_image(img_tensor, filename)
            generated_count += 1
        
        if generated_count % 100 == 0:
            print(f"Generated {generated_count}/{num_images} images...")

print(f"Successfully generated {num_images} images in '{output_dir}' folder!")

# Display 10 sample images in Jupyter notebook
print("Displaying 10 sample images...")
import matplotlib.pyplot as plt

sample_noise = torch.randn(10, nz, 1, 1, device=device)
with torch.no_grad():
    sample_images = netG(sample_noise)

# Create subplot grid
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    # Convert tensor to numpy and normalize
    img = sample_images[i].cpu()
    img = (img + 1) / 2  # Normalize from [-1,1] to [0,1]
    img = torch.clamp(img, 0, 1)
    img = img.permute(1, 2, 0).numpy()  # Change from CHW to HWC
    
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Sample {i+1}')

plt.tight_layout()
plt.show()

utils.save_image(sample_images, f"{output_dir}/sample_grid_10.png", normalize=True, nrow=5)
print("Sample images displayed above and saved as 'sample_grid_10.png'!")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from PIL import Image

# ============================
# IMPROVED SETTINGS
# ============================
dataset_path = "/Users/yusufabdulkadir/Downloads/CO2Wounds-V2 Extended Chronic Wounds Dataset From Leprosy Patients-2/imgs"
image_size = 128  # Increased resolution
batch_size = 32   # Smaller batch for better gradients
nz = 128          # Larger latent space
num_epochs = 200  # More epochs
lr_g = 0.0001     # Different learning rates
lr_d = 0.0004
beta1 = 0.0       # Updated beta values
beta2 = 0.9
ngpu = 1

# ============================
# IMPROVED DATASET WITH AUGMENTATION
# ============================
class ImageFolderNoClass(Dataset):
    def __init__(self, root_dir, transform=None):
        self.files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

# Enhanced data augmentation
transform = transforms.Compose([
    transforms.Resize((image_size + 16, image_size + 16)),  # Slightly larger
    transforms.RandomCrop(image_size),                       # Random crop
    transforms.RandomHorizontalFlip(p=0.5),                # Random flip
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 3-channel normalization
])

dataset = ImageFolderNoClass(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

# ============================
# IMPROVED MODELS WITH SPECTRAL NORMALIZATION
# ============================
def spectral_norm(module, use_spectral_norm=True):
    if use_spectral_norm:
        return nn.utils.spectral_norm(module)
    return module

def weights_init(m):
    """Custom weight initialization"""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class ImprovedGenerator(nn.Module):
    def __init__(self, nz=128, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            # Input: nz x 1 x 1 -> ngf*16 x 4 x 4
            nn.ConvTranspose2d(nz, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),
            
            # ngf*16 x 4 x 4 -> ngf*8 x 8 x 8
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            
            # ngf*8 x 8 x 8 -> ngf*4 x 16 x 16
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            
            # ngf*4 x 16 x 16 -> ngf*2 x 32 x 32
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            
            # ngf*2 x 32 x 32 -> ngf x 64 x 64
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            
            # ngf x 64 x 64 -> nc x 128 x 128
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

class ImprovedDiscriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            # Input: nc x 128 x 128 -> ndf x 64 x 64
            spectral_norm(nn.Conv2d(nc, ndf, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            
            # ndf x 64 x 64 -> ndf*2 x 32 x 32
            spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # ndf*2 x 32 x 32 -> ndf*4 x 16 x 16
            spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # ndf*4 x 16 x 16 -> ndf*8 x 8 x 8
            spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            # ndf*8 x 8 x 8 -> ndf*16 x 4 x 4
            spectral_norm(nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            
            # ndf*16 x 4 x 4 -> 1 x 1 x 1
            spectral_norm(nn.Conv2d(ndf * 16, 1, 4, 1, 0, bias=False)),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

# ============================
# INITIALIZATION AND TRAINING
# ============================
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

# Initialize models
netG = ImprovedGenerator(nz).to(device)
netD = ImprovedDiscriminator().to(device)

# Apply weight initialization
netG.apply(weights_init)
netD.apply(weights_init)

# Loss function with label smoothing
def label_smooth_loss(output, target_is_real, device, smooth=0.1):
    """Label smoothing for more stable training"""
    if target_is_real:
        # Real labels: 0.9 instead of 1.0
        target = torch.full_like(output, 1.0 - smooth, device=device)
    else:
        # Fake labels: 0.0 to 0.1 instead of 0.0
        target = torch.full_like(output, smooth, device=device)
    return nn.BCELoss()(output, target)

# Optimizers with different learning rates
optimizerD = optim.Adam(netD.parameters(), lr=lr_d, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr_g, betas=(beta1, beta2))

# Learning rate schedulers
schedulerD = optim.lr_scheduler.ExponentialLR(optimizerD, gamma=0.99)
schedulerG = optim.lr_scheduler.ExponentialLR(optimizerG, gamma=0.99)

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# ============================
# IMPROVED TRAINING LOOP
# ============================
print("Starting Improved Training Loop...")
G_losses = []
D_losses = []

for epoch in range(num_epochs):
    for i, real_images in enumerate(dataloader):
        real_images = real_images.to(device)
        b_size = real_images.size(0)

        # =============================
        # Train Discriminator
        # =============================
        netD.zero_grad()
        
        # Train on real images
        output_real = netD(real_images).view(-1)
        lossD_real = label_smooth_loss(output_real, True, device)
        lossD_real.backward()
        
        # Train on fake images
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_images = netG(noise)
        output_fake = netD(fake_images.detach()).view(-1)
        lossD_fake = label_smooth_loss(output_fake, False, device)
        lossD_fake.backward()
        
        optimizerD.step()
        
        # =============================
        # Train Generator (less frequently)
        # =============================
        if i % 2 == 0:  # Train G every other iteration
            netG.zero_grad()
            output = netD(fake_images).view(-1)
            lossG = label_smooth_loss(output, True, device)
            lossG.backward()
            optimizerG.step()
        else:
            lossG = torch.tensor(0.0)

        # Record losses
        D_losses.append((lossD_real + lossD_fake).item())
        G_losses.append(lossG.item())

        if i % 50 == 0:
            print(f"[{epoch}/{num_epochs}][{i}/{len(dataloader)}] "
                  f"Loss_D: {(lossD_real+lossD_fake).item():.4f} "
                  f"Loss_G: {lossG.item():.4f}")

    # Update learning rates
    schedulerD.step()
    schedulerG.step()
    
    # Save samples every 10 epochs
    if epoch % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        utils.save_image(fake, f"improved_epoch_{epoch}.png", normalize=True, nrow=8)

# Save final models
torch.save(netG.state_dict(), "improved_generator.pth")
torch.save(netD.state_dict(), "improved_discriminator.pth")
print("Improved training complete! Models saved.")

# Plot training losses
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(G_losses, label="Generator Loss")
plt.plot(D_losses, label="Discriminator Loss")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Losses")
plt.show()